# 📊 Notebook 1 — Data Preparation
**Formula 1 ML Analytics Project**

This notebook loads all 14 F1 CSV files from the Kaggle dataset, merges them into a single
master DataFrame, cleans the data, and saves `master_df.csv` for downstream notebooks.

> **Prerequisites**: Download the Kaggle dataset first. See `../data/README.md` for instructions.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Configuration — change DATA_PATH to where you placed the CSV files
# ---------------------------------------------------------------------------
DATA_PATH = "../data/raw/"

# Add src/ to path for imports
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "F1_ML_Project"))
sys.path.insert(0, "..")

print(f"Data path: {os.path.abspath(DATA_PATH)}")
print(f"Path exists: {os.path.exists(DATA_PATH)}")


In [ ]:
# Check which data files are present
EXPECTED_FILES = [
    "results.csv", "races.csv", "drivers.csv", "constructors.csv",
    "qualifying.csv", "pit_stops.csv", "lap_times.csv", "circuits.csv",
    "driver_standings.csv", "constructor_standings.csv", "status.csv",
    "constructor_results.csv", "sprint_results.csv", "seasons.csv"
]

if not os.path.exists(DATA_PATH):
    print("⚠️  Data directory not found!")
    print("Please follow the instructions in ../data/README.md to download the dataset.")
    print(f"Expected path: {os.path.abspath(DATA_PATH)}")
else:
    found, missing = [], []
    for f in EXPECTED_FILES:
        fp = os.path.join(DATA_PATH, f)
        if os.path.exists(fp):
            size_mb = os.path.getsize(fp) / 1e6
            found.append(f"  ✅ {f} ({size_mb:.1f} MB)")
        else:
            missing.append(f"  ❌ {f} — MISSING")
    print("Files found:")
    print("\n".join(found))
    if missing:
        print("\nFiles missing:")
        print("\n".join(missing))
        print("\n→ Download from: https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020")


In [ ]:
from src.data_loader import load_all_data, merge_master_df, clean_master_df, get_data_summary

# Load all available CSV files
data_dict = load_all_data(DATA_PATH)
print(f"\nLoaded {len(data_dict)} tables:")
for name, df in data_dict.items():
    print(f"  {name}: {df.shape[0]:,} rows × {df.shape[1]} cols")


In [ ]:
# Merge into master DataFrame
master_df = merge_master_df(data_dict)
print(f"Master DataFrame shape: {master_df.shape}")
master_df.head(3)


In [ ]:
# Clean master DataFrame
master_df = clean_master_df(master_df)
print("After cleaning:")
get_data_summary(master_df)


In [ ]:
# Inspect key columns
print("Sample rows — key columns:")
cols = ['year', 'round', 'driver_name', 'constructorName', 'grid', 'positionOrder', 
        'points', 'is_dnf', 'pit_lane_start', 'status']
available = [c for c in cols if c in master_df.columns]
master_df[available].head(10)


In [ ]:
# Data quality checks
print("=== DATA QUALITY REPORT ===")
print(f"\nTotal rows: {len(master_df):,}")
print(f"Seasons covered: {master_df['year'].min()} – {master_df['year'].max()}")
print(f"Unique drivers: {master_df['driverId'].nunique():,}")
print(f"Unique constructors: {master_df['constructorId'].nunique():,}")
print(f"Unique circuits: {master_df['circuitId'].nunique() if 'circuitId' in master_df.columns else 'N/A'}")
print(f"Total races: {master_df['raceId'].nunique():,}")
print(f"\nDNF rate: {master_df['is_dnf'].mean():.1%}" if 'is_dnf' in master_df.columns else "")
print(f"Pit lane starts: {master_df['pit_lane_start'].sum():,}" if 'pit_lane_start' in master_df.columns else "")
print(f"\nMissing qualifying data (pre-2003): {master_df['qual_position'].isna().sum():,}" if 'qual_position' in master_df.columns else "")
print(f"Missing lap time data (pre-1996): {master_df['mean_lap_time'].isna().sum():,}" if 'mean_lap_time' in master_df.columns else "")


In [ ]:
# Save master DataFrame
OUTPUT_PATH = "../data/processed/"
os.makedirs(OUTPUT_PATH, exist_ok=True)
out_file = os.path.join(OUTPUT_PATH, "master_df.csv")
master_df.to_csv(out_file, index=False)
print(f"✅ Saved master_df.csv → {out_file}")
print(f"   Shape: {master_df.shape}")
